# LogicAlpha Tsetlin Machine — private Tiingo benchmark

This notebook runs the frozen **development** benchmark on a Colab NVIDIA GPU. It does not request or store an API token. Upload the two licensed Tiingo files only into this private runtime, never into GitHub or a public notebook.

Before starting:

1. Open the Command Palette from **Tools → Command palette**.
2. Search for **Change runtime version** and select **26.07** (Python 3.12).
3. Select **Runtime → Change runtime type → T4 GPU**.

The current default Python 3.13 runtime cannot install the pinned `tmu==0.8.3` wheel. The final holdout remains locked.

In [ ]:
import sys
print('Python:', sys.version)
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Select Colab runtime version 26.07 (Python 3.12), reconnect, and run again.')
!nvidia-smi


## 1. Clone the public code and install pinned dependencies

Only public source code is cloned. The licensed market data are uploaded separately in the next step.

In [ ]:
import os
from pathlib import Path

if not Path('/content/logic-alpha-tm').exists():
    !git clone https://github.com/naibwedi/logic-alpha-tm.git /content/logic-alpha-tm
%cd /content/logic-alpha-tm
!python -m pip install -q -e ".[tm]" pycuda


## 2. Upload the two private input files

From your local project, select:

- `data/raw/tiingo-prices.csv`
- `data/raw/tiingo-prices.available-at.csv`

They remain in the temporary Colab runtime and disappear when the runtime is deleted. Do not share a runtime that still contains these files.

In [ ]:
import hashlib
from google.colab import files

uploaded = files.upload()
required = {'tiingo-prices.csv', 'tiingo-prices.available-at.csv'}
missing = required - set(uploaded)
if missing:
    raise ValueError(f'Missing required uploads: {sorted(missing)}')

raw_dir = Path('data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)
for name in required:
    content = uploaded[name]
    (raw_dir / name).write_bytes(content)
    print(name, len(content), 'bytes', hashlib.sha256(content).hexdigest())
del uploaded


## 3. Confirm CUDA and run the frozen development benchmark

The command records `CUDA` in `run-manifest.json`. If this cell reports a CUDA/PyCUDA error, use the local CPU workflow instead of silently changing the experiment.

In [ ]:
import pycuda.autoinit
import pycuda.driver as cuda
print('CUDA device:', cuda.Device(0).name())

!python -m logic_alpha_tm.cli benchmark \
  --csv data/raw/tiingo-prices.csv \
  --spec experiments/tiingo-v0.2.json \
  --phase development \
  --tmu-platform CUDA \
  --output results/tiingo-development-v0.2


## 4. Review and download the results

Download the result archive before ending the runtime. Do not run the locked holdout until the development methodology is frozen.

In [ ]:
import json
import shutil
from google.colab import files

result_dir = Path('results/tiingo-development-v0.2')
print((result_dir / 'BENCHMARK.md').read_text())
print(json.dumps(json.loads((result_dir / 'run-manifest.json').read_text()), indent=2))
archive = shutil.make_archive('/content/tiingo-development-v0.2', 'zip', result_dir)
files.download(archive)


## 5. Delete the private runtime

After the ZIP download finishes, choose **Runtime → Disconnect and delete runtime** to remove the uploaded licensed files from the Colab VM.